# CSV to JSON Converter




In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # CSV to JSON using Spark (Recommended)

# COMMAND ----------

# ===== CONFIGURATION (from job base_parameters or widgets) =====
dbutils.widgets.text("input_path", "/Volumes/dev_automotive/landing/landing_raw/catalogs.csv", "Input CSV path")
dbutils.widgets.text("output_path", "/Volumes/dev_automotive/landing/landing_raw/catalogs.json", "Output JSON path")
input_path = dbutils.widgets.get("input_path").strip()
output_path = dbutils.widgets.get("output_path").strip()

# COMMAND ----------

from pyspark.sql import functions as F

print("🔄 Converting CSV to JSON using Spark...")
print(f"Input:  {input_path}")
print(f"Output: {output_path}")
print()

# Read CSV with correct delimiter
df = spark.read \
    .option("header", "true") \
    .option("sep", ";") \
    .option("encoding", "UTF-8") \
    .option("mode", "PERMISSIVE") \
    .csv(input_path)

print(f"✅ Read {df.count():,} records")
print(f"✅ Columns: {len(df.columns)}")
print()

# Add audit columns
df = df \
    .withColumn("loaded_at", F.current_timestamp()) \
    .withColumn("source_file", F.lit(input_path))

# Write as single JSON file (coalesce to 1 partition)
df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .json(output_path + "_temp")

print("✅ JSON written to temp location")
print()

# Find the part file
part_files = [f.path for f in dbutils.fs.ls(output_path + "_temp") if f.name.startswith("part-")]

if part_files:
    # Move the part file to final location
    dbutils.fs.mv(part_files[0], output_path)
    
    # Clean up temp directory
    dbutils.fs.rm(output_path + "_temp", recurse=True)
    
    print(f"✅ Moved to: {output_path}")
else:
    print("❌ No part file found")

print()

# Verify
df_verify = spark.read.json(output_path)
print(f"✅ Verification: {df_verify.count():,} records")

print()
print("="*70)
print("✅ CONVERSION COMPLETE")
print("="*70)

display(df_verify.limit(5))